PART 1: API(Open-Meteo API)

In [1]:
import requests
import pandas as pd

cities_coords = {
    'London': {'lat': 51.5074, 'lon': -0.1278},
    'Paris': {'lat': 48.8566, 'lon': 2.3522},
    'Rome': {'lat': 41.9029, 'lon': 12.4964},
    'Berlin': {'lat': 52.5200, 'lon': 13.4050},
    'Madrid': {'lat': 40.4168, 'lon': -3.7038},
    'Vienna': {'lat': 48.2082, 'lon': 16.3738},
    'Amsterdam': {'lat': 52.3676, 'lon': 4.9041},
    'Prague': {'lat': 50.0755, 'lon': 14.4378},
    'Barcelona': {'lat': 41.3851, 'lon': 2.1734},
    'Istanbul': {'lat': 41.0082, 'lon': 28.9784}
}

weather_list = []

for city, coords in cities_coords.items():
    url = f"https://api.open-meteo.com/v1/forecast?latitude={coords['lat']}&longitude={coords['lon']}&current=temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        current = data.get('current', {})
        
        weather_list.append({
            'City': city,
            'Collection_Time': current.get('time'), # <--- Добавили время сбора из API
            'Temperature_C': current.get('temperature_2m'),
            'Humidity_pct': current.get('relative_humidity_2m'),
            'Precipitation_mm': current.get('precipitation'),
            'Windspeed_kmh': current.get('wind_speed_10m')
        })

df_weather = pd.DataFrame(weather_list)

df_weather.to_csv('weather_raw.csv', index=False)
print("Updated weather_raw.csv")

Updated weather_raw.csv


Part 2: Web Scraping (Wikipedia)

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

cities_list = ['London', 'Paris', 'Rome', 'Berlin', 'Madrid', 
               'Vienna', 'Amsterdam', 'Prague', 'Barcelona', 'Istanbul']

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
tourism_list = []

for city in cities_list:
    #example of a city attractions category URL
    wiki_url = f"https://en.wikipedia.org/wiki/Category:Tourist_attractions_in_{city}"
    resp = requests.get(wiki_url, headers=headers)
    
    if resp.status_code == 200:
        soup = BeautifulSoup(resp.text, 'html.parser')
        #finding page elements in the Wikipedia category
        category_groups = soup.find_all('div', class_='mw-category-group')
        
        attractions_count = 0
        for group in category_groups:
            links = group.find_all('a')
            attractions_count += len(links)
            
        tourism_list.append({
            'City': city,
            'Attractions_Count': attractions_count,
            'Source_URL': wiki_url
        })
    else:
        tourism_list.append({
            'City': city,
            'Attractions_Count': 0,
            'Source_URL': wiki_url
        })

#creating second DataFrame from Web Scraping 
df_tourism = pd.DataFrame(tourism_list)
print(f"Data of Tourism from Wikipedia:\n {df_tourism}")
df_tourism.to_csv('tourism_raw.csv', index=False)

Data of Tourism from Wikipedia:
         City  Attractions_Count  \
0     London                109   
1      Paris                115   
2       Rome                 90   
3     Berlin                 74   
4     Madrid                 32   
5     Vienna                 74   
6  Amsterdam                 44   
7     Prague                 65   
8  Barcelona                 78   
9   Istanbul                 62   

                                          Source_URL  
0  https://en.wikipedia.org/wiki/Category:Tourist...  
1  https://en.wikipedia.org/wiki/Category:Tourist...  
2  https://en.wikipedia.org/wiki/Category:Tourist...  
3  https://en.wikipedia.org/wiki/Category:Tourist...  
4  https://en.wikipedia.org/wiki/Category:Tourist...  
5  https://en.wikipedia.org/wiki/Category:Tourist...  
6  https://en.wikipedia.org/wiki/Category:Tourist...  
7  https://en.wikipedia.org/wiki/Category:Tourist...  
8  https://en.wikipedia.org/wiki/Category:Tourist...  
9  https://en.wikipedia.org/wik